In [4]:
# Requirements: pandas, numpy
# Usage: set IN_PATH and OUT_PATH, then run.

import pandas as pd
import numpy as np
from pathlib import Path

# ---- Configuration (edit as needed)
IN_PATH  = Path("data/iam_raw.csv")                 # your input file
OUT_PATH = Path("electricity_mix_2025_2100.csv")
REGION_FILTER   = "World"                      # e.g., "World"
SCENARIO_FILTER = None                         # e.g., "SSP2-26" or None to keep all
MODEL_FILTER    = None                         # e.g., "IMAGE 3.0.1" or None to keep all
VARIABLE_PREFIX = "Secondary Energy|Electricity|"  # electricity variables in IAMC format
TARGET_YEARS = np.arange(2025, 2101)           # 2025–2100 inclusive

# ---- Load
df = pd.read_csv(IN_PATH)

# Basic cleaning: keep only rows that look like real data
df = df.dropna(subset=["Variable", "Region"])

# Optional filters
if REGION_FILTER is not None:
    df = df[df["Region"].astype(str).str.strip().eq(REGION_FILTER)]
if SCENARIO_FILTER is not None:
    df = df[df["Scenario"].astype(str).str.strip().eq(SCENARIO_FILTER)]
if MODEL_FILTER is not None:
    df = df[df["Model"].astype(str).str.strip().eq(MODEL_FILTER)]

# Keep electricity-related variables
elec = df[df["Variable"].astype(str).str.startswith(VARIABLE_PREFIX)].copy()
if elec.empty:
    raise ValueError("No electricity rows found. Check VARIABLE_PREFIX/filters.")

# Extract source name after the last '|'
elec["Source"] = elec["Variable"].astype(str).str.split("|").str[-1].str.strip()

# Identify numeric year columns (IAMC wide format)
year_cols = [c for c in elec.columns if str(c).isdigit()]
if not year_cols:
    raise ValueError("No year columns detected (e.g., '2020', '2030', ...).")

# Melt to long format
melt = elec.melt(
    id_vars=["Model", "Scenario", "Region", "Source", "Unit"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value",
)
melt["Year"] = pd.to_numeric(melt["Year"], errors="coerce")
melt["Value"] = pd.to_numeric(melt["Value"], errors="coerce")
melt = melt.dropna(subset=["Year", "Source", "Value"])

# Aggregate duplicates if any (same Model/Scenario/Region/Source/Year)
melt = (melt
        .groupby(["Model", "Scenario", "Region", "Source", "Unit", "Year"], as_index=False)["Value"]
        .sum())

# Compute shares (%) within each (Model, Scenario, Region, Year)
melt["Total"] = melt.groupby(["Model", "Scenario", "Region", "Year"])["Value"].transform("sum")
melt["Share_pct"] = (melt["Value"] / melt["Total"].replace(0, np.nan)) * 100.0

# Build annual interpolation 2025–2100 for each (Model, Scenario, Region)
out_frames = []
for (model, scen, region), sub in melt.groupby(["Model", "Scenario", "Region"], dropna=False):
    wide = sub.pivot(index="Year", columns="Source", values="Share_pct").sort_index()
    # Insert missing target years and interpolate linearly along the index
    years_union = sorted(set(wide.index).union(TARGET_YEARS))
    wide_interp = (wide
                   .reindex(years_union)
                   .interpolate(method="index", limit_direction="both")
                   .loc[TARGET_YEARS])

    # Renormalize rows to exactly 100% (protect against small drift)
    wide_interp = wide_interp.div(wide_interp.sum(axis=1), axis=0) * 100.0
    wide_interp = wide_interp.round(4)

    # Attach identifiers
    wide_interp.insert(0, "Year", TARGET_YEARS)
    wide_interp.insert(1, "Model", model)
    wide_interp.insert(2, "Scenario", scen)
    wide_interp.insert(3, "Region", region)
    out_frames.append(wide_interp.reset_index(drop=True))

result = pd.concat(out_frames, ignore_index=True)

# If there’s only one (Model, Scenario, Region), drop those cols for a cleaner file
if result[["Model","Scenario","Region"]].drop_duplicates().shape[0] == 1:
    result = result.drop(columns=["Model","Scenario","Region"])

# Save
result.to_csv(OUT_PATH, index=False)
print(f"Wrote: {OUT_PATH.resolve()}")


Wrote: /Users/farshidnazemi/pack-repo/packaging-roadmap/electricity_mix_2025_2100.csv


In [5]:
import pandas as pd
import numpy as np

# ===== Input: decadal absolute values in million metric tonnes =====
IN_CSV = "data/ssp2_baseline_oil_gas_biomass.csv"

# ===== 1) Load and standardize column names =====
df = pd.read_csv(IN_CSV)

def find_col(df, keys):
    for c in df.columns:
        cl = c.lower()
        if any(k in cl for k in keys):
            return c
    raise ValueError(f"Missing a column matching: {keys}")

year_col = find_col(df, ["year"])
oil_col  = find_col(df, ["oil"])
gas_col  = find_col(df, ["natural gas", "nat gas", "gas"])
bio_col  = find_col(df, ["biomass"])

df = df.rename(columns={
    year_col: "Year",
    oil_col: "Oil_Mt",
    gas_col: "NaturalGas_Mt",
    bio_col: "Biomass_Mt"
})
for c in ["Year", "Oil_Mt", "NaturalGas_Mt", "Biomass_Mt"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# ===== 2) Interpolate to annual values =====
df = df.set_index("Year").sort_index()
annual_index = pd.Index(range(int(df.index.min()), int(df.index.max()) + 1), name="Year")
df_annual = df.reindex(annual_index).interpolate(method="linear")

# Focus on 2025–2100
df_yr = df_annual.loc[2025:2100].copy()

# ===== 3) Shares (%) of each source over the sum of the three =====
df_yr["Total_Mt"] = df_yr[["Oil_Mt", "NaturalGas_Mt", "Biomass_Mt"]].sum(axis=1)
den_all = df_yr["Total_Mt"].replace(0, np.nan)

shares = pd.DataFrame({
    "Oil_share_%":        (df_yr["Oil_Mt"]        / den_all) * 100.0,
    "NaturalGas_share_%": (df_yr["NaturalGas_Mt"] / den_all) * 100.0,
    "Biomass_share_%":    (df_yr["Biomass_Mt"]    / den_all) * 100.0,
}, index=df_yr.index).round(2)

shares.to_csv("ssp2_baseline_shares_2025_2100.csv", index_label="Year")

# ===== 4) Ratio of oil to (oil + natural gas) =====
den_fossil = (df_yr["Oil_Mt"] + df_yr["NaturalGas_Mt"]).replace(0, np.nan)
oil_vs_gas = pd.DataFrame({
    "Oil_over_(Oil+Gas)_fraction": (df_yr["Oil_Mt"] / den_fossil),
    "Oil_over_(Oil+Gas)_percent":  (df_yr["Oil_Mt"] / den_fossil) * 100.0
}, index=df_yr.index).round(4)

oil_vs_gas.to_csv("ratio_oil_over_oil_plus_gas_2025_2100.csv", index_label="Year")

# ===== 5) Ratio of biomass to (oil + natural gas) =====
bio_vs_fossil = pd.DataFrame({
    "Biomass_over_(Oil+Gas)_fraction": (df_yr["Biomass_Mt"] / den_fossil),
    "Biomass_over_(Oil+Gas)_percent":  (df_yr["Biomass_Mt"] / den_fossil) * 100.0
}, index=df_yr.index).round(4)

bio_vs_fossil.to_csv("ratio_biomass_over_oil_plus_gas_2025_2100.csv", index_label="Year")

# Quick sanity check printouts (optional)
print(shares.head())
print(oil_vs_gas.head())
print(bio_vs_fossil.head())
print("Wrote:",
      "ssp2_baseline_shares_2025_2100.csv,",
      "ratio_oil_over_oil_plus_gas_2025_2100.csv,",
      "ratio_biomass_over_oil_plus_gas_2025_2100.csv")

      Oil_share_%  NaturalGas_share_%  Biomass_share_%
Year                                                  
2025        85.56               14.44              0.0
2026        85.22               14.78              0.0
2027        84.89               15.11              0.0
2028        84.58               15.42              0.0
2029        84.29               15.71              0.0
      Oil_over_(Oil+Gas)_fraction  Oil_over_(Oil+Gas)_percent
Year                                                         
2025                       0.8556                     85.5556
2026                       0.8522                     85.2174
2027                       0.8489                     84.8936
2028                       0.8458                     84.5833
2029                       0.8429                     84.2857
      Biomass_over_(Oil+Gas)_fraction  Biomass_over_(Oil+Gas)_percent
Year                                                                 
2025                              0.0    

In [7]:
import pandas as pd
import numpy as np

IN_CSV = "data/ssp2_baseline_oil_gas_biomass.csv"

# 1) Load and standardize
df = pd.read_csv(IN_CSV)

def find_col(keys):
    for c in df.columns:
        cl = c.lower()
        if any(k in cl for k in keys):
            return c
    raise ValueError(f"Missing column matching: {keys}")

year_col = find_col(["year"])
oil_col  = find_col(["oil"])
gas_col  = find_col(["natural gas", "nat gas", "gas"])
bio_col  = find_col(["biomass"])

df = df.rename(columns={
    year_col: "Year",
    oil_col: "Oil_Mt",
    gas_col: "NaturalGas_Mt",
    bio_col: "Biomass_Mt",
}).astype({"Year": int, "Oil_Mt": float, "NaturalGas_Mt": float, "Biomass_Mt": float})

# 2) Interpolate to annual values and subset to 2025–2100
df = df.set_index("Year").sort_index()
annual_index = pd.Index(range(df.index.min(), df.index.max() + 1), name="Year")
dfA = df.reindex(annual_index).interpolate(method="linear").loc[2025:2100].copy()

# ----- A) Oil vs Natural Gas (exclude Biomass) -----
den_fossil_2 = (dfA["Oil_Mt"] + dfA["NaturalGas_Mt"]).replace(0, np.nan)

oil_vs_gas = pd.DataFrame({
    "Oil":        (dfA["Oil_Mt"]        / den_fossil_2),
    "NaturalGas": (dfA["NaturalGas_Mt"] / den_fossil_2)
}, index=dfA.index).round(2)

# sanity: sums to ~100
oil_vs_gas["Sum"] = (oil_vs_gas["Oil"] + oil_vs_gas["NaturalGas_%"]).round(2)

oil_vs_gas.to_csv("ratio_oil_vs_natural_gas_2025_2100.csv", index_label="Year")

# ----- B) Biomass share of TOTAL (bio vs. fossil) -----
total = (dfA["Oil_Mt"] + dfA["NaturalGas_Mt"] + dfA["Biomass_Mt"]).replace(0, np.nan)

bio_vs_total = pd.DataFrame({
    "Biomass": (dfA["Biomass_Mt"] / total),
    "Fossil":  ((dfA["Oil_Mt"] + dfA["NaturalGas_Mt"]) / total)
}, index=dfA.index).round(2)

# sanity: sums to ~100
bio_vs_total["Sum"] = (bio_vs_total["Biomass"] + bio_vs_total["Fossil"]).round(2)

bio_vs_total.to_csv("ratio_biomass_vs_total_2025_2100.csv", index_label="Year")


Wrote:
 - ratio_oil_vs_natural_gas_2025_2100.csv  (Oil_% + NaturalGas_% = 100)
 - ratio_biomass_vs_total_2025_2100.csv    (Biomass_% + Fossil_% = 100)
